# Step 1 — Required libraries install/import

In [1]:
# Task 4: Sentiment Analysis

!pip -q install nltk

import pandas as pd
import nltk

from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

# Step 2 — Review Dataset

In [16]:
# Save the review dataset

df_reviews.to_csv(
    "reviews.csv",
    index=False
)

print("reviews.csv created successfully!")

reviews.csv created successfully!


In [17]:
from google.colab import files

files.download("reviews.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Step 3 — Dataset Inspection

In [18]:
# Step 3: Dataset Inspection

print("Dataset Shape:", df_reviews.shape)

print("\nColumn Names:")
print(df_reviews.columns.tolist())

print("\nData Types:")
print(df_reviews.dtypes)

print("\nMissing Values:")
print(df_reviews.isnull().sum())

print("\nDuplicate Rows:")
print(df_reviews.duplicated().sum())

print("\nDataset Information:")
df_reviews.info()

Dataset Shape: (30, 4)

Column Names:
['Review', 'Cleaned_Review', 'Sentiment_Score', 'Sentiment']

Data Types:
Review              object
Cleaned_Review      object
Sentiment_Score    float64
Sentiment           object
dtype: object

Missing Values:
Review             0
Cleaned_Review     0
Sentiment_Score    0
Sentiment          0
dtype: int64

Duplicate Rows:
0

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Review           30 non-null     object 
 1   Cleaned_Review   30 non-null     object 
 2   Sentiment_Score  30 non-null     float64
 3   Sentiment        30 non-null     object 
dtypes: float64(1), object(3)
memory usage: 1.1+ KB


In [19]:
df_reviews


,Review,Cleaned_Review,Sentiment_Score,Sentiment
0,This product is absolutely amazing and I love it,this product is absolutely amazing and i love it,0.8610,Positive
1,Very good quality and excellent product,very good quality and excellent product,0.7841,Positive
2,I am really happy with this purchase,i am really happy with this purchase,0.6115,Positive
3,The product works perfectly,the product works perfectly,0.6369,Positive
4,"Amazing experience, highly recommended",amazing experience highly recommended,0.7089,Positive
5,Good product for the price,good product for the price,0.4404,Positive
6,I really liked this product,i really liked this product,0.4754,Positive
7,Excellent quality and fast delivery,excellent quality and fast delivery,0.5719,Positive
8,Very satisfied with my purchase,very satisfied with my purchase,0.4754,Positive
9,The product is wonderful,the product is wonderful,0.5719,Positive


# Step 4 — Text Cleaning

In [20]:
# Step 4: Text Cleaning

import re

def clean_text(text):
    text = text.lower()                         # lowercase
    text = re.sub(r"http\S+|www\S+", "", text)  # remove URLs
    text = re.sub(r"[^a-zA-Z\s]", "", text)     # remove special characters
    text = re.sub(r"\s+", " ", text).strip()    # remove extra spaces
    return text

df_reviews["Cleaned_Review"] = df_reviews["Review"].apply(clean_text)

df_reviews.head()

,Review,Cleaned_Review,Sentiment_Score,Sentiment
0,This product is absolutely amazing and I love it,this product is absolutely amazing and i love it,0.8610,Positive
1,Very good quality and excellent product,very good quality and excellent product,0.7841,Positive
2,I am really happy with this purchase,i am really happy with this purchase,0.6115,Positive
3,The product works perfectly,the product works perfectly,0.6369,Positive
4,"Amazing experience, highly recommended",amazing experience highly recommended,0.7089,Positive


In [21]:
print("Original Review:")
print(df_reviews["Review"].iloc[0])

print("\nCleaned Review:")
print(df_reviews["Cleaned_Review"].iloc[0])

Original Review:
This product is absolutely amazing and I love it

Cleaned Review:
this product is absolutely amazing and i love it


# Step 5 — Sentiment Analysis

In [22]:
# Step 5: Sentiment Analysis

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

# Calculate sentiment score
df_reviews["Sentiment_Score"] = df_reviews["Cleaned_Review"].apply(
    lambda text: sia.polarity_scores(text)["compound"]
)

# Classify sentiment
def classify_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df_reviews["Sentiment"] = df_reviews["Sentiment_Score"].apply(
    classify_sentiment
)

# Display results
df_reviews

,Review,Cleaned_Review,Sentiment_Score,Sentiment
0,This product is absolutely amazing and I love it,this product is absolutely amazing and i love it,0.8610,Positive
1,Very good quality and excellent product,very good quality and excellent product,0.7841,Positive
2,I am really happy with this purchase,i am really happy with this purchase,0.6115,Positive
3,The product works perfectly,the product works perfectly,0.6369,Positive
4,"Amazing experience, highly recommended",amazing experience highly recommended,0.7089,Positive
5,Good product for the price,good product for the price,0.4404,Positive
6,I really liked this product,i really liked this product,0.4754,Positive
7,Excellent quality and fast delivery,excellent quality and fast delivery,0.5719,Positive
8,Very satisfied with my purchase,very satisfied with my purchase,0.4754,Positive
9,The product is wonderful,the product is wonderful,0.5719,Positive


In [23]:
print("Sentiment Distribution:")
print(df_reviews["Sentiment"].value_counts())

print("\nAverage Sentiment Score:",
      round(df_reviews["Sentiment_Score"].mean(), 3))

Sentiment Distribution:
Sentiment
Positive    13
Negative    12
Neutral      5
Name: count, dtype: int64

Average Sentiment Score: 0.069


# Step 6 — Sentiment Visualization

### Chart 1 — Positive / Neutral / Negative Reviews

In [24]:
# Step 6: Sentiment Distribution

import plotly.express as px

sentiment_counts = df_reviews["Sentiment"].value_counts()

fig = px.bar(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    text=sentiment_counts.values,
    title="Sentiment Distribution of Reviews",
    labels={
        "x": "Sentiment",
        "y": "Number of Reviews"
    }
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    template="plotly_dark",
    height=500
)

fig.show()

### Chart 2 — Sentiment Score Distribution

In [25]:
# Sentiment Score Distribution

fig = px.histogram(
    df_reviews,
    x="Sentiment_Score",
    nbins=10,
    title="Distribution of Sentiment Scores",
    labels={
        "Sentiment_Score": "Sentiment Score",
        "count": "Number of Reviews"
    }
)

fig.update_layout(
    template="plotly_dark",
    height=500
)

fig.show()

### Bonus — Pie Chart

In [26]:
# Sentiment Percentage

fig = px.pie(
    df_reviews,
    names="Sentiment",
    title="Overall Sentiment Percentage",
    hole=0.35
)

fig.update_layout(
    template="plotly_dark",
    height=500
)

fig.show()

#Step 7: Key Insights & Final Conclusion

### 7.1 Sentiment Summary

In [27]:
# Step 7: Sentiment Summary

sentiment_counts = df_reviews["Sentiment"].value_counts()

total_reviews = len(df_reviews)

sentiment_percentage = (
    df_reviews["Sentiment"].value_counts(normalize=True) * 100
).round(2)

print("Total Reviews:", total_reviews)

print("\nSentiment Counts:")
print(sentiment_counts)

print("\nSentiment Percentage:")
print(sentiment_percentage)

print("\nAverage Sentiment Score:",
      round(df_reviews["Sentiment_Score"].mean(), 3))

Total Reviews: 30

Sentiment Counts:
Sentiment
Positive    13
Negative    12
Neutral      5
Name: count, dtype: int64

Sentiment Percentage:
Sentiment
Positive    43.33
Negative    40.00
Neutral     16.67
Name: proportion, dtype: float64

Average Sentiment Score: 0.069


## 7.2 Important Insights

## Key Insights

1. The sentiment analysis classifies the reviews into Positive, Neutral, and Negative categories.

2. The sentiment distribution shows the overall emotional tendency of the review dataset.

3. Positive reviews indicate customer satisfaction with the product or experience.

4. Negative reviews indicate dissatisfaction, poor experience, or product-related problems.

5. Neutral reviews represent average or factual opinions without a strong positive or negative emotion.

6. The sentiment score helps measure the strength of the emotion expressed in each review.

7. The average sentiment score provides an overall indication of the sentiment of the collected sample reviews.

## Business Insights

- Positive feedback can help identify features or aspects that customers appreciate.
- Negative feedback can help identify areas where product quality or customer experience can be improved.
- Neutral feedback can provide information about basic customer expectations.
- Sentiment analysis can support businesses in understanding customer feedback more efficiently.

## 7.3 Final Conclusion

## Final Conclusion

Sentiment Analysis was performed on a sample dataset of product reviews using Python and NLTK's VADER sentiment analyzer.

The reviews were first cleaned and then analyzed using sentiment scores. Based on the compound sentiment score, each review was classified as Positive, Neutral, or Negative.

Visualizations were created using Plotly to understand the sentiment distribution and sentiment score distribution.

The analysis demonstrates how Natural Language Processing and sentiment analysis can be used to convert textual customer feedback into meaningful insights.

Overall, this project provided practical experience in text preprocessing, sentiment scoring, classification, visualization, and interpretation of customer reviews.

## Internship Task

**CodeAlpha Data Analytics Internship – Task 4: Sentiment Analysis**